# Data Manipulation with SQL
This course will tackle the following
- Conditional
- Subqueries
- Window Functions

## SQL CASE Statement: The Data Engineer's Guide
The real important thing to know here is this: **`CASE`- `END` returns a column**

### 1. The Core Concept

The `CASE` statement is SQL's way of handling `IF-THEN-ELSE` logic. It evaluates conditions sequentially and returns the first match.

### 2. Syntax Reference

**Standard Syntax (Range Based)**
Used for comparing different conditions (e.g., greater than, less than).

```sql
CASE
    WHEN column_name > 100 THEN 'High'
    WHEN column_name > 50  THEN 'Medium'
    ELSE 'Low' -- Optional, but recommended. Defaults to NULL if omitted.
END

```

**Binary Syntax (Exact Match)**
Used when checking a specific value equality (cleaner for simple lookups).

```sql
CASE department_id
    WHEN 1 THEN 'HR'
    WHEN 2 THEN 'Engineering'
    ELSE 'Contractor'
END

```

---

### 3. The Three Key Patterns

#### **Pattern A: The "Labeler" (Categorization)**

* **Use it when:** You need to group raw numbers into human-readable buckets.
* **Where:** Inside the `SELECT` clause.

```sql
SELECT
    order_id,
    CASE
        WHEN DATEDIFF(day, order_date, GETDATE()) > 30 THEN 'Overdue'
        WHEN status = 'Paid' THEN 'Completed'
        ELSE 'Active'
    END AS invoice_status
FROM Orders;

```

#### **Pattern B: The "Pivoter" (Conditional Aggregation)**

* **Use it when:** You need to turn row values into column headers, or compare subsets of data side-by-side.
* **Where:** Inside `SUM()` or `COUNT()`.

```sql
SELECT
    -- Count rows that match a condition
    SUM(CASE WHEN status = 'Error' THEN 1 ELSE 0 END) AS error_count,

    -- Sum values based on a condition
    SUM(CASE WHEN category = 'Electronics' THEN revenue ELSE 0 END) AS electronics_revenue
FROM Sales;

```

* *Tip:* Prefer `SUM(... 1 ELSE 0)` over `COUNT` to avoid NULL confusion.

#### **Pattern C: The "Dynamic Filter"**

* **Use it when:** You want a query to change its filtering logic based on a user input or variable.
* **Where:** Inside the `WHERE` clause.

```sql
-- Filter by status, or return ALL if input is 'All'
SELECT *
FROM Employees
WHERE (
    CASE
        WHEN @input_var = 'All' THEN 1  -- If 'All', condition is True (1=1)
        WHEN status = @input_var THEN 1 -- If match, condition is True
        ELSE 0
    END
) = 1;

```

---

### 4. Common Pitfalls & Best Practices

| Issue | Description | Solution |
| --- | --- | --- |
| **Overlapping Logic** | SQL stops at the *first* true condition. | Order your `WHEN` clauses from most specific to least specific. |
| **Null Aggregates** | `SUM` ignores NULLs. If no rows match, result is NULL. | Use `ELSE 0` inside the CASE to ensure you get `0` instead of `NULL`. |
| **Data Types** | All `THEN` results must be the same type. | Do not mix Strings and Integers (e.g., `THEN 'High' ... THEN 1`). |
| **Performance** | CASE in `WHERE` clauses can be slow (Non-Sargable). | For massive tables, prefer standard `OR` logic over `CASE` in filters. |


### 5. Developing the "Sense": When to Pivot?

Knowing *how* to write the code is easy; knowing *when* you need it is the engineering skill. Use these triggers to recognize when a standard `GROUP BY` isn't enough and you need **Conditional Aggregation (`SUM(CASE...)`)**.

#### **Trigger A: The "Values-to-Columns" Shape**

Look at the shape of your raw data versus the shape of the report requested.

* **Raw Data:** The variable is in a *row* (e.g., a column named `Status` has values 'Success', 'Fail').
* **The Request:** The variable is a *column header* (e.g., "I need a column for Success Count and a column for Fail Count").
* **The Rule:** If you need to turn a **value** into a **header**, you must pivot.

#### **Trigger B: The "One Row Per X" Constraint**

Your stakeholder says: *"I want one row per Customer, showing their Total Purchases and Total Refunds."*

* **The Problem:** In your database, Purchases and Refunds are on separate rows.
* **The Intuition:** You cannot use a simple `WHERE type = 'Purchase'` because that filters out the refunds. You need to keep *all* rows but sort them into different buckets using `CASE` inside the `SUM`.

#### **Trigger C: The "Side-by-Side" Math**

Your stakeholder says: *"Calculate the Success Rate (Successes / Total Events) per day."*

* **The Math:** To divide `A / B`, both numbers must exist on the **same row**.
* **The Intuition:** If "Successes" and "Failures" are currently on different rows, you cannot divide them. You must first pivot them into columns on the same row, *then* divide.

#### **The "Finger Test" (Practical Check)**

1. Put your finger on a single row of your raw data.
2. Ask: *"Does this one row contain all the numbers I need for the final report?"*
3. **If NO** (e.g., you are looking at a 'Purchase' row but you also need 'Refund' info to calculate the total), you almost certainly need to use **Conditional Aggregation**.
